# 🌐 Global AI Layoffs — Exploratory Data Analysis & Visualization
**Dataset:** Global AI Layoffs and Job Market (2020–Present)  

---

## 📌 Analysis Areas
1. Goal 1: Which industries were most affected?
2. Goal 2: How did layoffs trend over time?
3. Goal 3: Are AI companies more resilient?
4. Goal 4: Which cities/countries had the highest reductions?
5. Goal 5: Which funding stages are most vulnerable?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from dash import Dash, html, dcc, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Libraries loaded ✅")

Libraries loaded ✅


In [3]:
df = pd.read_csv("../data/cleaned_layoff_dataset.csv", parse_dates=["date"])
print(f"Shape: {df.shape}")
df.head()

Shape: (2470, 17)


,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company,layoff_count_missing,raised_mm_missing,year,month,quarter,year_quarter
0,Panda Squad,Sf Bay Area,6.0,2020-03-13,75.0,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,1.0,United States,False,False,False,2020,3,1,2020Q1
1,Hopskipdrive,Los Angeles,8.0,2020-03-13,10.0,Transportat…,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,45.0,United States,False,False,False,2020,3,1,2020Q1
2,Help.Com,Austin,16.0,2020-03-16,100.0,Support,LinkedIn,Seed,6.0,United States,False,False,False,2020,3,1,2020Q1
3,Service,Los Angeles,109.0,2020-03-16,100.0,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,5.0,United States,False,True,False,2020,3,1,2020Q1
4,Inspirato,Denver,130.0,2020-03-16,22.0,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,79.0,United States,False,False,False,2020,3,1,2020Q1


## 1. Amgad: Goal 1
Which industries were most affected?

In [7]:
df['year'] = df['date'].dt.year.astype(str)

unique_years = sorted(df['year'].unique())

# 2. Extract All-Time data
all_time_data = df.groupby('industry')['layoff_count'].sum().reset_index()
all_time_data = all_time_data[all_time_data['layoff_count'] > 0]

# 3. Create the Base Treemap
fig = px.treemap(
  all_time_data,
  path=[px.Constant("All Industry Layoffs"), 'industry'],
  values='layoff_count',
  color='layoff_count',
  color_continuous_scale='Blues',
  template='plotly_white'
)

# 4. Build the dynamic menu buttons
buttons = []

buttons.append(dict(
 method="restyle",
 label="All Years",
 args=[{
 "values": [all_time_data['layoff_count']],
 "labels": [all_time_data['industry']],
 "marker.colors": [all_time_data['layoff_count']] # Forces color to recalculate properly
 }]
))

for yr in unique_years:
 year_df = df[df['year'] == yr].groupby('industry')['layoff_count'].sum().reset_index()
 year_df = year_df[year_df['layoff_count'] > 0]

 buttons.append(dict(
 method="restyle",
 label=f"Year: {yr}",
 args=[{
 "values": [year_df['layoff_count']],
 "labels": [year_df['industry']],
 "marker.colors": [year_df['layoff_count']] # FIX: Forces color range to reset for 2025!
 }]
 ))

# 5. Fix Layout Layout and Titles
fig.update_layout(
 # FIX: Position title higher up and give it padding
 title={
 'text': '<b>Industry Impact Assessment: Total Layoffs by Sector</b>',
 'y': 0.96,
 'x': 0.01,
 'xanchor': 'left',
 'yanchor': 'top'
 },
 # FIX: Move dropdown down slightly so it sits safely below the title
 updatemenus=[dict(
 buttons=buttons,
 direction="down",
 showactive=True,
 x=0.01,
 xanchor="left",
 y=1.05,
 yanchor="bottom"
 )],
 margin=dict(t=120, b=20, l=20, r=20), # Gives breathing room for title + filter
 height=650,
 coloraxis_showscale=False
)

fig.update_traces(textinfo="label+value", textfont=dict(size=14))
fig.show()

In [ ]:
# 2. Extract All-Time data (grouped by BOTH industry and company)
all_time_data = df.groupby(['industry', 'company'])['layoff_count'].sum().reset_index()
all_time_data = all_time_data[all_time_data['layoff_count'] > 0]

# 3. Create the Base Treemap with the new multi-layered path
fig = px.treemap(
    all_time_data,
    path=[px.Constant("All Industry Layoffs"), 'industry', 'company'], # Added 'company' to the path hierarchy
    values='layoff_count',
    color='layoff_count',
    color_continuous_scale='Blues',
    template='plotly_white'
)

# 4. Build the dynamic menu buttons using method="update"
buttons = []

# Base option: All Years
buttons.append(dict(
    method="update", # 'update' alters both data AND layout settings seamlessly
    label="All Years",
    args=[
        {
            "values": [all_time_data['layoff_count']],
            # When using path hierarchies, Plotly relies on ids/labels calculated internally.
            # Passing None let's Plotly recalculate the deep tree layout automatically.
            "labels": [None],
            "marker.colors": [all_time_data['layoff_count']]
        },
        {"title.text": "<b>Industry & Company Impact Assessment: Total Layoffs (All Years)</b>"}
    ]
))

# Generate separate data hierarchies for each individual year
for yr in unique_years:
    year_df = df[df['year'] == yr].groupby(['industry', 'company'])['layoff_count'].sum().reset_index()
    year_df = year_df[year_df['layoff_count'] > 0]

    # We build a temporary standalone figure to extract its pre-computed tree structures
    temp_fig = px.treemap(
        year_df,
        path=[px.Constant("All Industry Layoffs"), 'industry', 'company'],
        values='layoff_count'
    )

    buttons.append(dict(
        method="update",
        label=f"Year: {yr}",
        args=[
            {
                "ids": [temp_fig.data[0].ids],
                "parents": [temp_fig.data[0].parents],
                "labels": [temp_fig.data[0].labels],
                "values": [temp_fig.data[0].values],
                "marker.colors": [temp_fig.data[0].values]
            },
            {"title.text": f"<b>Industry & Company Impact Assessment: Total Layoffs ({yr})</b>"}
        ]
    ))

# 5. Fix Layout and Adjust Top Menus
fig.update_layout(
    title={
        'text': '<b>Industry & Company Impact Assessment: Total Layoffs</b>',
        'y': 0.96,
        'x': 0.01,
        'xanchor': 'left',
        'yanchor': 'top'
    },
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.01,
        xanchor="left",
        y=1.05,
        yanchor="bottom"
    )],
    margin=dict(t=130, b=20, l=20, r=20), # Slightly expanded top margin for text space
    height=700, # Increased height to make reading text inside smaller company blocks easier
    coloraxis_showscale=False
)

# Text configurations
fig.update_traces(
    textinfo="label+value",
    textfont=dict(size=13)
)

fig.show()

In [37]:
industry = (
    df.groupby("industry")["layoff_count"]
    .sum()
    .nlargest(10)
    .reset_index()
)
industry.columns = ["Industry", "Total Layoffs"]

fig = px.bar(
    industry, x="Total Layoffs", y="Industry",
    orientation="h",
    title="Top 10 Industries by Total Layoffs",
    color="Total Layoffs",
    color_continuous_scale="Teal"
)
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5",
    coloraxis_showscale=False
)
fig.show()

In [38]:
companies = (
    df.groupby("company")["layoff_count"]
    .sum()
    .nlargest(15)
    .reset_index()
)
companies.columns = ["Company", "Total Layoffs"]

fig = px.bar(
    companies, x="Total Layoffs", y="Company",
    orientation="h",
    title="Top 15 Companies by Total Layoffs",
    color="Total Layoffs",
    color_continuous_scale="Teal"
)
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5",
    coloraxis_showscale=False
)
fig.show()

## 2. Aditi: Goal 2
How did layoffs trend over time?

In [9]:
# 2. AGGREGATE
# Create a dataframe grouped by Quarter for broader trend visibility
quarterly_df = df.groupby(pd.Grouper(key='date', freq='QE'))['layoff_count'].sum().reset_index()



# 3. CALCULATE YoY CHANGE (Example for Monthly)
monthly_df = df.groupby(pd.Grouper(key='date', freq='ME'))['layoff_count'].sum().reset_index()
monthly_df['year'] = monthly_df['date'].dt.year
monthly_df['month'] = monthly_df['date'].dt.month
# Compare current month to same month last year
monthly_df['YoY_Change'] = monthly_df.groupby(['month'])['layoff_count'].pct_change()



# 4. PLOTTING (Google Trends Style)
fig = px.line(
    monthly_df,
    x='date',
    y='layoff_count',
    title='Monthly Layoff Trends',
    markers=True
)



fig.update_layout(
    hovermode='x unified',        # Provides the clean, vertical interactive line
    plot_bgcolor='white',
    xaxis=dict(
        title="Year (2020 - 2026)",
        tickformat="%Y",             # Displays only the year
        dtick="M12",                 # Forces a tick mark every 12 months
        showgrid=True,
        gridcolor='lightgray',
        rangeslider=dict(visible=False)
    ),
    yaxis=dict(showgrid=True, gridcolor='lightgray')

)



fig.show()

In [35]:
yearly = df.groupby("year")["layoff_count"].sum().reset_index()
yearly.columns = ["Year", "Total Layoffs"]

fig = px.bar(
    yearly, x="Year", y="Total Layoffs",
    title="Total Layoffs by Year (2020–2026)",
    color="Total Layoffs",
    color_continuous_scale="Teal",
    text="Total Layoffs"
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5", coloraxis_showscale=False)
fig.show()

In [36]:
quarterly = df.groupby("year_quarter")["layoff_count"].sum().reset_index()
quarterly.columns = ["Quarter", "Total Layoffs"]

fig = px.line(
    quarterly, x="Quarter", y="Total Layoffs",
    title="Quarterly Layoff Trend (2020–2026)",
    markers=True,
    color_discrete_sequence=["#01696f"]
)
fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor="#f9f8f5",
    paper_bgcolor="#f9f8f5"
)
fig.show()

## 3. Sarvenaz: Goal 3
Are AI companies more resilient?

In [10]:
# Group AI and Non-AI companies
ai_vs_nonai = df.groupby('is_ai_company')['pct_workforce'].mean().reset_index()

# Rename labels for readability
ai_vs_nonai['company_type'] = ai_vs_nonai['is_ai_company'].map({
    True: 'AI Companies',
    False: 'Non-AI Companies'
})

# Create visualization
fig = px.bar(
    ai_vs_nonai,
    x='company_type',
    y='pct_workforce',
    color='company_type',
    text='pct_workforce',
    title='Average Workforce Reduction: AI vs Non-AI Companies',
    labels={
        'company_type': 'Company Type',
        'pct_workforce': 'Average % Workforce Laid Off'
    }
)

# Improve appearance
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')

fig.update_layout(
    title_x=0.5,
    height=550,
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='#EAF2F8',
    showlegend=False,
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    )
)

fig.show()

In [11]:
full_shutdown = df.copy()

full_shutdown['full_reduction'] = full_shutdown['pct_workforce'] == 100

full_shutdown['company_type'] = full_shutdown['is_ai_company'].map({
    True: 'AI Companies',
    False: 'Non-AI Companies'
})

full_shutdown['year'] = pd.to_datetime(full_shutdown['date']).dt.year
years = sorted(full_shutdown['year'].dropna().unique())

fig = go.Figure()

# 🎨 COLOR MAP (MAIN UPGRADE)
color_map = {
    'AI Companies': '#636EFA',        # blue
    'Non-AI Companies': '#EF553B'     # red
}

# ALL YEARS
all_data = full_shutdown.groupby('company_type')['full_reduction'].mean().reset_index()
all_data['full_reduction'] *= 100

fig.add_trace(
    go.Bar(
        x=all_data['company_type'],
        y=all_data['full_reduction'],
        marker=dict(
            color=[color_map[x] for x in all_data['company_type']],
            line=dict(color='black', width=0.6)
        ),
        text=[f"{v:.1f}%" for v in all_data['full_reduction']],
        textposition='outside',
        name='All Years',
        hovertemplate='%{x}<br>%{y:.2f}% full reduction<extra></extra>'
    )
)

# YEARLY DATA
for year in years:
    year_data = full_shutdown[full_shutdown['year'] == year]

    summary = year_data.groupby('company_type')['full_reduction'].mean().reset_index()
    summary['full_reduction'] *= 100

    fig.add_trace(
        go.Bar(
            x=summary['company_type'],
            y=summary['full_reduction'],
            marker=dict(
                color=[color_map[x] for x in summary['company_type']],
                line=dict(color='black', width=0.6)
            ),
            text=[f"{v:.1f}%" for v in summary['full_reduction']],
            textposition='outside',
            name=str(year),
            visible=False,
            hovertemplate='%{x}<br>%{y:.2f}% full reduction<extra></extra>'
        )
    )

# DROPDOWN
buttons = []

buttons.append(
    dict(
        label='📊 All Years',
        method='update',
        args=[
            {'visible': [True] + [False]*len(years)},
            {'title': 'Full Workforce Reduction (%) — All Years'}
        ]
    )
)

for i, year in enumerate(years):
    visibility = [False] * (len(years) + 1)
    visibility[i + 1] = True

    buttons.append(
        dict(
            label=f'📅 {year}',
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'Full Workforce Reduction (%) — {year}'}
            ]
        )
    )

# LAYOUT IMPROVEMENTS
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            x=1.15,
            y=1.15,
            showactive=True,
            bgcolor='white',
            bordercolor='lightgray',
            font=dict(size=12)
        )
    ],

    title=dict(
        text='Full Workforce Reduction (%) — AI vs Non-AI Companies',
        x=0.5,
        font=dict(size=18)
    ),

    height=600,

    plot_bgcolor='white',
    paper_bgcolor='#F5F7FB',

    yaxis=dict(
        title='% Companies with 100% Workforce Reduction',
        gridcolor='lightgray',
        zeroline=True
    ),

    xaxis=dict(
        title='Company Type',
        tickfont=dict(size=12)
    ),

    bargap=0.35,
    showlegend=False
)

fig.show()

In [45]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Avg % Workforce Laid Off", "Total Layoff Events"])

fig.add_trace(go.Bar(
    x=ai_compare["Company Type"],
    y=ai_compare["Avg % Workforce Laid Off"],
    marker_color=["#01696f", "#4f98a3"],
    name="Avg % Workforce"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=ai_compare["Company Type"],
    y=ai_compare["Layoff Events"],
    marker_color=["#01696f", "#4f98a3"],
    name="Layoff Events"
), row=1, col=2)

fig.update_layout(
    title="AI vs Non-AI Company Layoff Comparison",
    plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5",
    showlegend=False
)
fig.show()

In [49]:
# Filter AI companies only, get top 5 by total layoffs
top5_ai = (
    df[df['is_ai_company'] == True]
    .groupby('company')['layoff_count']
    .sum()
    .nlargest(5)
    .reset_index()
)
top5_ai.columns = ['Company', 'Total Layoffs']

fig = px.bar(
    top5_ai,
    x='Total Layoffs',
    y='Company',
    orientation='h',
    title='Top 5 AI Companies by Total Layoffs',
    color='Total Layoffs',
    color_continuous_scale='Teal',
    text='Total Layoffs'
)

fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    coloraxis_showscale=False,
    title_font_size=18,
    width=900,
    height=500,
    xaxis_title='Total Layoffs',
    yaxis_title='Company'
)

fig.show()

In [58]:
# Step 1 — aggregate first
top5_both = (
    df.groupby(['is_ai_company', 'company'])['layoff_count']
    .sum()
    .reset_index()
)

# Step 2 — map label
top5_both['is_ai_company'] = top5_both['is_ai_company'].map({True: 'AI Company', False: 'Non-AI Company'})

# Step 3 — split, get top 5 each, then concat back together
ai     = top5_both[top5_both['is_ai_company'] == 'AI Company'].nlargest(5, 'layoff_count')
non_ai = top5_both[top5_both['is_ai_company'] == 'Non-AI Company'].nlargest(5, 'layoff_count')
final  = pd.concat([ai, non_ai]).reset_index(drop=True)

fig2 = px.bar(
    final,
    x='layoff_count',
    y='company',
    color='is_ai_company',
    orientation='h',
    facet_col='is_ai_company',
    facet_col_spacing=0.12,
    title='Top 5 Companies by Layoffs — AI vs Non-AI',
    labels={
        'layoff_count'  : 'Total Layoffs',
        'company'       : 'Company',
        'is_ai_company' : 'Company Type'
    },
    color_discrete_map={
        'AI Company'    : '#01696f',
        'Non-AI Company': '#4f98a3'
    },
    text='layoff_count'
)

fig2.update_traces(texttemplate='%{text:,.0f}', textposition='outside')

fig2.update_yaxes(matches=None, showticklabels=True)
fig2.update_xaxes(matches=None, showticklabels=True)

fig2.update_layout(
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    showlegend=False,
    title_font_size=18,
    width=1100,
    height=500
)

fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.for_each_yaxis(lambda y: y.update(title=''))
fig2.show()

In [59]:
# Use pct_workforce as the resilience metric — lower = more resilient
resilient = (
    df.groupby(['is_ai_company', 'company'])['pct_workforce']
    .mean()
    .reset_index()
)

resilient['is_ai_company'] = resilient['is_ai_company'].map(
    {True: 'AI Company', False: 'Non-AI Company'}
)

# Get bottom 5 (lowest % laid off) per group
ai_low     = resilient[resilient['is_ai_company'] == 'AI Company'].nsmallest(5, 'pct_workforce')
non_ai_low = resilient[resilient['is_ai_company'] == 'Non-AI Company'].nsmallest(5, 'pct_workforce')
resilient_final = pd.concat([ai_low, non_ai_low]).reset_index(drop=True)

fig = px.bar(
    resilient_final,
    x='pct_workforce',
    y='company',
    color='is_ai_company',
    orientation='h',
    facet_col='is_ai_company',
    facet_col_spacing=0.12,
    title='Top 5 Most Resilient Companies — AI vs Non-AI (Lowest % Workforce Laid Off)',
    labels={
        'pct_workforce'  : '% Workforce Laid Off',
        'company'        : 'Company',
        'is_ai_company'  : 'Company Type'
    },
    color_discrete_map={
        'AI Company'     : '#01696f',
        'Non-AI Company' : '#4f98a3'
    },
    text='pct_workforce'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None, showticklabels=True)

fig.update_layout(
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    showlegend=False,
    title_font_size=16,
    width=1100,
    height=500
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.for_each_yaxis(lambda y: y.update(title=''))
fig.show()

In [60]:
# Use pct_workforce as the resilience metric — lower = more resilient
resilient = (
    df.groupby(['is_ai_company', 'company'])['layoff_count']
    .mean()
    .reset_index()
)

resilient['is_ai_company'] = resilient['is_ai_company'].map(
    {True: 'AI Company', False: 'Non-AI Company'}
)

# Get bottom 5 (lowest % laid off) per group
ai_low     = resilient[resilient['is_ai_company'] == 'AI Company'].nsmallest(5, 'layoff_count')
non_ai_low = resilient[resilient['is_ai_company'] == 'Non-AI Company'].nsmallest(5, 'layoff_count')
resilient_final = pd.concat([ai_low, non_ai_low]).reset_index(drop=True)

fig = px.bar(
    resilient_final,
    x='layoff_count',
    y='company',
    color='is_ai_company',
    orientation='h',
    facet_col='is_ai_company',
    facet_col_spacing=0.12,
    title='Top 5 Most Resilient Companies — AI vs Non-AI (Lowest % Workforce Laid Off)',
    labels={
        'layoff_count'  : 'Total Layoffs',
        'company'        : 'Company',
        'is_ai_company'  : 'Company Type'
    },
    color_discrete_map={
        'AI Company'     : '#01696f',
        'Non-AI Company' : '#4f98a3'
    },
    text='layoff_count'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None, showticklabels=True)

fig.update_layout(
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    showlegend=False,
    title_font_size=16,
    width=1100,
    height=500
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.for_each_yaxis(lambda y: y.update(title=''))
fig.show()

## 4. Albin & Sadia: Goal 4
Which cities/countries had the highest reductions?

In [12]:
#total layoff by countries
country_layoffs = (
    df.groupby('country')['layoff_count']
      .sum()
      .sort_values(ascending=False)
)

print(country_layoffs)

country
United States     426694.5
India              39216.0
Germany            28732.0
United Kingdo…     21710.0
Netherlands        20240.5
Sweden             18207.5
Israel             10656.0
Canada              8604.0
China               5769.0
Brazil              5732.0
Australia           5404.5
Singapore           5287.5
Japan               4335.0
Indonesia           2980.0
Nigeria             2598.0
France              1938.5
Kenya               1102.5
Cayman Islands      1060.0
Austria              810.0
Mexico               779.0
Spain                691.0
Poland               610.0
Ireland              585.0
Finland              577.5
Switzerland          566.0
Czech Republic       566.0
Norway               525.5
United Arab E…       459.0
Denmark              455.0
Argentina            453.5
Colombia             424.5
Uruguay              404.0
Russia               400.0
South Korea          395.0
Estonia              364.0
Hong Kong            340.0
Saudi Arabia        

In [13]:
df['country'].unique()

<StringArray>
[ 'United States',         'Israel',          'India',         'Canada',
      'Singapore', 'United Kingdo…',         'Brazil',        'Estonia',
      'Indonesia',        'Denmark',      'Hong Kong',         'Mexico',
      'Australia',        'Germany',        'Nigeria',   'South Africa',
        'Vietnam',    'Netherlands',         'France',        'Ireland',
          'China',         'Sweden',         'Russia',      'Argentina',
       'Pakistan',         'Turkey',        'Romania', 'United Arab E…',
       'Colombia',        'Finland',           'Peru',     'Luxembourg',
          'Kenya',        'Austria',        'Senegal',    'New Zealand',
     'Seychelles',          'Ghana',        'Hungary',       'Malaysia',
        'Belgium',          'Spain',            'Uae',          'Egypt',
    'Switzerland',          'Chile',       'Portugal',          'Italy',
    'South Korea',          'Japan',         'Poland',        'Ukraine',
    'Philippines',   'Saudi Arabia', 

In [14]:
country_mapping = {
    'US': 'United States',
    'UK': 'United Kingdom',
    'UAE': 'United Arab Emirates',
    'South Korea': 'Korea, South',
    'Russia': 'Russian Federation'
}

df['country_clean'] = df['country'].replace(country_mapping)
# Aggregate layoffs by country
country_df = (
    df.groupby(['year', 'country_clean'])['layoff_count']
      .sum()
      .reset_index()
)

# Create choropleth map
fig = px.choropleth(
    country_df,
    locations='country_clean',
    locationmode='country names',
    color='layoff_count',
    hover_name='country_clean',
    color_continuous_scale='Reds',
    animation_frame='year',
    title='Total Layoffs by Country'
)
# Convert to globe projection
fig.update_geos(
    projection_type="orthographic",
    showcoastlines=True,
    showland=True,
    showcountries=True
)

fig.update_layout(
    width=800,
    height=650,
    margin=dict(l=0, r=0, t=50, b=0)
)
fig.update_geos(
    projection_type='orthographic',
    showcoastlines=True,
    showland=True,
    showcountries=True
)
fig.show()

C:\Users\amgad\AppData\Local\Temp\ipykernel_24696\1004719969.py:18: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [15]:
#total layoff with top 10 cities
df_clean = df.copy()
top_cities = (
    df_clean.groupby('location')['layoff_count']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()   
)

print(top_cities)

             location  layoff_count
0         Sf Bay Area      188241.5
1             Seattle       79762.0
2              Austin       27719.5
3       New York City       26764.0
4          Sacramento       22801.5
5  Bengaluru Non-U.S.       20623.5
6     London Non-U.S.       20337.0
7  Amsterdam Non-U.S.       18540.5
8  Stockholm Non-U.S.       17905.0
9              Boston       16515.5


In [16]:
df_clean = df.copy()

year_city = (
    df_clean.groupby(['year', 'location'])['layoff_count']
      .sum()
      .reset_index()
)

# rank per year
year_city['rank'] = year_city.groupby('year')['layoff_count'] \
                             .rank(method='first', ascending=False)

top_yearly = year_city[year_city['rank'] <= 10]
top_yearly = top_yearly.sort_values(['year', 'layoff_count'], ascending=[True, True])
fig = px.bar(
    top_yearly,
    x='layoff_count',
    y='location',
    color='location',
    animation_frame='year',
    orientation='h',
    category_orders={
        "location": top_yearly.groupby("location")["layoff_count"].sum()
                       .sort_values(ascending=False).index.tolist()
    },
    title='Top 10 Cities by Layoffs Each Year'
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})

fig.show()

In [26]:
#Geo spatial
# Determine which cities and countries are experiencing the highest workforce reductions to support recruitment and workforce planning strategies.

import ipywidgets as widgets
from IPython.display import display, HTML

# Ensure 'date' column is datetime and 'year' column exists.
# This step is crucial for filtering the data by year.
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# Re-initialize geolocator and geocode if not already available for robustness.
# This helps prevent issues if the kernel restarts or previous geopy setup was incomplete.
try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    geolocator = Nominatim(user_agent="layoff_analysis_app")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1) # Respect Nominatim's usage policy
except ImportError:
    print("geopy not found. Please ensure it's installed (e.g., !pip install geopy).")



In [27]:

# Define the function to get geocoded cities for a given year.
# This function identifies top cities and converts their names to coordinates.
def get_geocoded_cities_for_year(data_frame, selected_year):
    df_year = data_frame[data_frame['year'] == selected_year]
    top_10_cities_layoffs_year = df_year.groupby('location')['layoff_count'].sum().nlargest(10).reset_index()

    city_coords_year = []
    for index, row in top_10_cities_layoffs_year.iterrows():
        city_name = row['location']
        layoff_count = row['layoff_count']

        # Use country information to improve geocoding accuracy for cities.
        country_for_city = data_frame[data_frame['location'] == city_name]['country'].iloc[0] if not data_frame[data_frame['location'] == city_name].empty else ''
        try:
            location = geocode(f"{city_name}, {country_for_city}")
            if location:
                city_coords_year.append({
                    'city': city_name,
                    'layoff_count': layoff_count,
                    'latitude': location.latitude,
                    'longitude': location.longitude
                })
        except Exception as e:
            print(f"Could not geocode {city_name} for year {selected_year}: {e}")

    return pd.DataFrame(city_coords_year)

# Define the function to create a Folium map for the geocoded cities of a specific year.
def create_folium_city_map_for_year(geocoded_df_to_plot, selected_year):
    if geocoded_df_to_plot.empty:
        m = folium.Map(location=[0, 0], zoom_start=1, tiles='CartoDB positron', scrollWheelZoom=False, dragging=False, zoom_control=True)
        title_html = f'''<h3 align="center" style="font-size:20px"><b>No Data for Top Cities in {selected_year}</b></h3>'''
        m.get_root().html.add_child(folium.Element(title_html))
        return m

    avg_lat = geocoded_df_to_plot['latitude'].mean()
    avg_lon = geocoded_df_to_plot['longitude'].mean()
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=2, tiles='CartoDB positron',
                   scrollWheelZoom=False, dragging=False, zoom_control=True)

    title_html = f'''
                 <h3 align="center" style="font-size:20px"><b>Top 10 Cities by Layoff Count in {selected_year}</b></h3>
                 '''
    m.get_root().html.add_child(folium.Element(title_html))

    for index, row in geocoded_df_to_plot.iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"<b>{row['city']}</b><br>Total Layoffs: {int(row['layoff_count'])}",
            tooltip=row['city']
        ).add_to(m)
    return m

# Get available years from your DataFrame to set the slider range.
available_years = sorted(df['year'].unique())

# Create an IntSlider widget for year selection.
year_slider = widgets.IntSlider(
    value=min(available_years) if available_years else 2020, # Default to the earliest year or 2020
    min=min(available_years) if available_years else 2020,
    max=max(available_years) if available_years else 2023, # Default to the latest year or 2023
    step=1,
    description='Select Year:',
    continuous_update=False, # Only update map when slider is released for better performance
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Define the main update function that will be linked to the slider.
# This function will fetch data, create the map, and display it.
def update_map_with_slider(selected_year):
    geocoded_cities_for_year_df = get_geocoded_cities_for_year(df, selected_year)
    folium_map = create_folium_city_map_for_year(geocoded_cities_for_year_df, selected_year)
    display(folium_map)

# Display the slider and link it to the update function using ipywidgets.interactive.
print("Use the slider below to select a year:")
widgets.interactive(update_map_with_slider, selected_year=year_slider)


Use the slider below to select a year:


interactive(children=(IntSlider(value=2020, continuous_update=False, description='Select Year:', max=2026, min…

In [28]:
# 1. Determine which cities are experiencing the highest workforce reductions
import ipywidgets as widgets
from IPython.display import display, HTML

# Ensure 'date' column is datetime and 'year' column exists.
# This step is crucial for filtering the data by year.
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# Re-initialize geolocator and geocode if not already available for robustness.
# This helps prevent issues if the kernel restarts or previous geopy setup was incomplete.
try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    geolocator = Nominatim(user_agent="layoff_analysis_app")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1) # Respect Nominatim's usage policy
except ImportError:
    print("geopy not found. Please ensure it's installed (e.g., !pip install geopy).")

# Define the function to get geocoded cities for a given year.
# This function identifies top cities and converts their names to coordinates.
def get_geocoded_cities_for_year(data_frame, selected_year):
    df_year = data_frame[data_frame['year'] == selected_year]
    top_10_cities_layoffs_year = df_year.groupby('location')['layoff_count'].sum().nlargest(10).reset_index()

    city_coords_year = []
    for index, row in top_10_cities_layoffs_year.iterrows():
        city_name = row['location']
        layoff_count = row['layoff_count']

        # Use country information to improve geocoding accuracy for cities.
        country_for_city = data_frame[data_frame['location'] == city_name]['country'].iloc[0] if not data_frame[data_frame['location'] == city_name].empty else ''
        try:
            location = geocode(f"{city_name}, {country_for_city}")
            if location:
                city_coords_year.append({
                    'city': city_name,
                    'layoff_count': layoff_count,
                    'latitude': location.latitude,
                    'longitude': location.longitude
                })
        except Exception as e:
            print(f"Could not geocode {city_name} for year {selected_year}: {e}")

    return pd.DataFrame(city_coords_year)

# Define the function to create a Folium map for the geocoded cities of a specific year.
def create_folium_city_map_for_year(geocoded_df_to_plot, selected_year):
    if geocoded_df_to_plot.empty:
        m = folium.Map(location=[0, 0], zoom_start=1, tiles='CartoDB positron', scrollWheelZoom=False, dragging=False, zoom_control=True)
        title_html = f'''<h3 align="center" style="font-size:20px"><b>No Data for Top Cities in {selected_year}</b></h3>'''
        m.get_root().html.add_child(folium.Element(title_html))
        return m

    avg_lat = geocoded_df_to_plot['latitude'].mean()
    avg_lon = geocoded_df_to_plot['longitude'].mean()
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=2, tiles='CartoDB positron',
                   scrollWheelZoom=False, dragging=False, zoom_control=True)

    title_html = f'''
                 <h3 align="center" style="font-size:20px"><b>Top 10 Cities by Layoff Count in {selected_year}</b></h3>
                 '''
    m.get_root().html.add_child(folium.Element(title_html))

    for index, row in geocoded_df_to_plot.iterrows():
        # Calculate radius for CircleMarker based on layoff count
        radius_value = (row['layoff_count'] ** 0.5) / 5

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius_value,
            popup=f"<b>{row['city']}</b><br>Total Layoffs: {int(row['layoff_count'])}",
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.4
        ).add_to(m)
    return m

# Get available years from your DataFrame to set the slider range.
available_years = sorted(df['year'].unique())

# Create an IntSlider widget for year selection.
year_slider = widgets.IntSlider(
    value=min(available_years) if available_years else 2020, # Default to the earliest year or 2020
    min=min(available_years) if available_years else 2020,
    max=max(available_years) if available_years else 2023, # Default to the latest year or 2023
    step=1,
    description='Select Year:',
    continuous_update=False, # Only update map when slider is released for better performance
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Define the main update function that will be linked to the slider.
# This function will fetch data, create the map, and display it.
def update_map_with_slider(selected_year):
    geocoded_cities_for_year_df = get_geocoded_cities_for_year(df, selected_year)
    folium_map = create_folium_city_map_for_year(geocoded_cities_for_year_df, selected_year)
    display(folium_map)

# Display the slider and link it to the update function using ipywidgets.interactive.
print("Use the slider below to select a year and view the corresponding top cities by layoff count:")
widgets.interactive(update_map_with_slider, selected_year=year_slider)

Use the slider below to select a year and view the corresponding top cities by layoff count:


interactive(children=(IntSlider(value=2020, continuous_update=False, description='Select Year:', max=2026, min…

In [41]:
country = df.groupby("country")["layoff_count"].sum().reset_index()
country.columns = ["Country", "Total Layoffs"]

fig = px.choropleth(
    country, locations="Country", locationmode="country names",
    color="Total Layoffs",
    title="Global Layoff Distribution by Country",
    color_continuous_scale="Teal",
    labels={"Total Layoffs": "Total Laid Off"}
)
fig.update_layout(paper_bgcolor="#f9f8f5")
fig.show()

C:\Users\amgad\AppData\Local\Temp\ipykernel_24696\2570981357.py:4: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [42]:
cities = (
    df.groupby("location")["layoff_count"]
    .sum()
    .nlargest(10)
    .reset_index()
)
cities.columns = ["City", "Total Layoffs"]

fig = px.bar(
    cities, x="City", y="Total Layoffs",
    title="Top 10 Cities by Total Layoffs",
    color="Total Layoffs",
    color_continuous_scale="Teal",
    text="Total Layoffs"
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(
    xaxis_tickangle=-30,
    plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5",
    coloraxis_showscale=False
)
fig.show()

## 5. Saeeda & Harpreet: Goal 5
Which funding stages are most vulnerable?

In [17]:
stage_summary=df.groupby('stage').agg(Total_layoffs=('layoff_count',"sum"),average_layoffs=("layoff_count", "mean"),layoff_events=("layoff_count", "count")).sort_values("layoff_events", ascending=False).reset_index()
stage_summary

,stage,Total_layoffs,average_layoffs,layoff_events
0,Post-Ipo,368294.0,656.495544,561
1,Unknown,60225.5,146.178398,412
2,Series B,28587.5,104.333942,274
3,Series C,21234.0,89.218487,238
4,Acquired,53786.0,242.279279,222
5,Series D,20316.5,99.104878,205
6,Series A,12284.5,68.628492,179
7,Series E,18825.5,174.310185,108
8,Seed,6998.0,76.901099,91
9,Series F,9680.5,142.360294,68


In [32]:
fig = px.scatter(
    stage_summary,
    x='layoff_events',
    y='average_layoffs',
    size='Total_layoffs',
    color='stage',
    title='Funding Stage Risk: Layoff Frequency vs Average Layoffs',
    labels={
        'layoff_events'   : 'Layoff Frequency (Number of Events)',
        'average_layoffs' : 'Average Layoffs per Event',
        'Total_layoffs'   : 'Total Layoffs',
        'stage'           : 'Funding Stage'
    },
    hover_name='stage',
    hover_data={
        'Total_layoffs'   : ':,.0f',
        'average_layoffs' : ':,.1f',
        'layoff_events'   : True,
        'stage'           : False
    },
    size_max=80,
    template='plotly_white',
    text='stage'           # ← FIX 1: pass text INSIDE px.scatter, not update_traces
)

fig.update_traces(
    textposition='top center',
    textfont=dict(size=11),
    mode='markers+text'    # ← FIX 2: no text= here anymore
)

fig.update_layout(
    title_font_size=18,
    legend_title_text='Funding Stage',
    xaxis_title='Layoff Frequency (Number of Events)',
    yaxis_title='Average Layoffs per Event',
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    width=1200,            # ← FIX 3: make chart wider
    height=700,            # ← FIX 3: make chart taller
    legend=dict(
        font=dict(size=12),
        itemsizing='constant'
    )
)

fig.show()


# Top-right bubbles → high frequency AND high average = most dangerous stages

# Large bubbles → stages responsible for the most total layoffs overall

# Bottom-left bubbles → rare events with smaller average impact

In [ ]:
fig = px.scatter(
    stage_summary,
    x='average_layoffs',       # ← swapped
    y='layoff_events',         # ← swapped
    size='Total_layoffs',
    color='stage',
    title='Funding Stage Risk: Average Layoffs vs Layoff Frequency',
    labels={
        'average_layoffs' : 'Average Layoffs per Event',
        'layoff_events'   : 'Layoff Frequency (Number of Events)',
        'Total_layoffs'   : 'Total Layoffs',
        'stage'           : 'Funding Stage'
    },
    hover_name='stage',
    hover_data={
        'Total_layoffs'   : ':,.0f',
        'average_layoffs' : ':,.1f',
        'layoff_events'   : True,
        'stage'           : False
    },
    size_max=80,
    template='plotly_white',
    text='stage'
)

fig.update_traces(
    textposition='top center',
    textfont=dict(size=11),
    mode='markers+text'
)

fig.update_layout(
    title_font_size=18,
    legend_title_text='Funding Stage',
    xaxis_title='Average Layoffs per Event',     # ← swapped
    yaxis_title='Layoff Frequency (Number of Events)',  # ← swapped
    plot_bgcolor='#f9f8f5',
    paper_bgcolor='#f9f8f5',
    width=1200,
    height=700,
    legend=dict(
        font=dict(size=12),
        itemsizing='constant'
    )
)

fig.show()


# High on Y-axis → happens very frequently

# Far right on X-axis → large average layoffs per event when it does happen

# Top-right → frequent AND large = highest risk stages

# Bottom-right → rare but devastating when they happen (e.g. Series H, J)

# Top-left → frequent but smaller cuts each time (e.g. Seed, Series A)

In [48]:
fig = px.scatter(
    stage, x="Event_Count", y="Avg_Layoffs",
    size="Total_Layoffs", color="stage",
    title="Funding Stage Risk: Avg Layoffs vs Event Frequency",
    labels={"Event_Count": "Number of Layoff Events", "Avg_Layoffs": "Avg Layoffs per Event"},
    hover_name="stage",
    size_max=60,
    color_discrete_sequence=px.colors.sequential.Teal
)
fig.update_layout(plot_bgcolor="#f9f8f5", paper_bgcolor="#f9f8f5")
fig.show()

In [18]:
import plotly.express as px

fig = px.scatter(
    stage_summary,
    x="stage",
    y="average_layoffs",
    size="average_layoffs",
    color="layoff_events",
    text="stage",
    title="Funding Stage Vulnerability",
    labels={
        "stage": "Funding Stage",
        "average_layoffs": "Average Layoffs",
        "layoff_events": "Number of Layoff Events"
    },
    color_continuous_scale="Rainbow",
    size_max=80
)

fig.update_traces(
    textposition="top center",
    marker=dict(
        line=dict(width=2, color="black"),
        opacity=0.85
    )
)

fig.update_layout(
    title_x=0.5,
    height=650,

    # Background colors
    plot_bgcolor="#F3F6FF",
    paper_bgcolor="#EAF2F8",

    # Grid lines
    xaxis=dict(
        title="Funding Stage",
        showgrid=True,
        gridcolor="lightgray",
        tickangle=-45
    ),
    yaxis=dict(
        title="Average Layoffs",
        showgrid=True,
        gridcolor="lightgray"
    ),

    # Font styling
    font=dict(
        size=13
    )
)

fig.show()

In [20]:
# Sort for clean animation order
stage_summary_sorted = stage_summary.sort_values("average_layoffs").copy()

# Preserve stage order
stage_order = stage_summary_sorted["stage"].tolist()

# Create cumulative animation frames
frames = []

for i in range(1, len(stage_summary_sorted) + 1):
    temp = stage_summary_sorted.iloc[:i].copy()
    temp["frame"] = f"Step {i}"
    frames.append(temp)

animated_stage_summary = pd.concat(frames, ignore_index=True)

# Identify the most vulnerable funding stage
top_stage = stage_summary_sorted.loc[
    stage_summary_sorted["average_layoffs"].idxmax()
]

# Premium professional color scale
premium_scale = [
    [0.0, "#DCEEFF"],
    [0.25, "#8ECAE6"],
    [0.50, "#219EBC"],
    [0.75, "#126782"],
    [1.0, "#023047"]
]

fig = px.scatter(
    animated_stage_summary,
    x="stage",
    y="average_layoffs",
    size="average_layoffs",
    color="layoff_events",
    text="stage",
    animation_frame="frame",
    category_orders={"stage": stage_order},
    title="Funding Stage Vulnerability",
    labels={
        "stage": "Funding Stage",
        "average_layoffs": "Average Layoffs",
        "layoff_events": "Number of Layoff Events"
    },
    color_continuous_scale=premium_scale,
    size_max=95,
    range_y=[0, stage_summary["average_layoffs"].max() * 1.25],
    range_color=[
        stage_summary["layoff_events"].min(),
        stage_summary["layoff_events"].max()
    ]
)

fig.update_traces(
    textposition="top center",
    marker=dict(
        line=dict(width=1.8, color="#0F172A"),
        opacity=0.90
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Average Layoffs: %{y:,.0f}<br>"
        "Layoff Events: %{marker.color:,.0f}<br>"
        "<extra></extra>"
    )
)

fig.update_layout(
    title=dict(
        text=(
            "Funding Stage Vulnerability"
            "<br><sup>Bubble size shows average layoffs, while color intensity shows layoff event frequency</sup>"
        ),
        x=0.5,
        font=dict(size=25, family="Arial Black", color="#0F172A")
    ),

    height=720,
    width=1150,

    plot_bgcolor="#F8FBFD",
    paper_bgcolor="#EAF4F8",

    font=dict(
        family="Arial",
        size=13,
        color="#1E293B"
    ),

    xaxis=dict(
        title="Funding Stage",
        showgrid=False,
        tickangle=-40,
        showline=True,
        linewidth=1.5,
        linecolor="#94A3B8",
        ticks="outside"
    ),

    yaxis=dict(
        title="Average Layoffs",
        showgrid=True,
        gridcolor="#D8E3EA",
        zeroline=False,
        showline=True,
        linewidth=1.5,
        linecolor="#94A3B8",
        ticks="outside"
    ),

    coloraxis_colorbar=dict(
        title="Layoff Events",
        thickness=18,
        len=0.75
    ),

    transition=dict(
        duration=800,
        easing="cubic-in-out"
    ),

    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Arial",
        font_color="#0F172A"
    ),

    margin=dict(t=110, l=70, r=70, b=90),

    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.1,
            y=1.15,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        {
                            "frame": {"duration": 850, "redraw": True},
                            "transition": {"duration": 500},
                            "fromcurrent": True
                        }
                    ]
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }
                    ]
                )
            ]
        )
    ]
)

fig.add_annotation(
    x=top_stage["stage"],
    y=top_stage["average_layoffs"],
    text=f"Highest Avg Layoffs<br><b>{top_stage['stage']}</b>",
    showarrow=True,
    arrowhead=2,
    arrowsize=1.2,
    arrowwidth=2,
    arrowcolor="#023047",
    ax=40,
    ay=-60,
    bgcolor="rgba(255,255,255,0.95)",
    bordercolor="#8ECAE6",
    borderwidth=1.5,
    font=dict(size=12, color="#023047")
)

fig.show()

In [22]:
# Create grouped stage column

stage_group_mapping = {
    "Seed": "Early Stage",
    "Series A": "Early Stage",
    "Series B": "Early Stage",

    "Series C": "Late Stage & Growth",
    "Series D": "Late Stage & Growth",
    "Series E": "Late Stage & Growth",
    "Series F": "Late Stage & Growth",
    "Series G": "Late Stage & Growth",
    "Series H": "Late Stage & Growth",
    "Series I": "Late Stage & Growth",
    "Series J": "Late Stage & Growth",

    "Post-Ipo": "Exited / Public",
    "Post-IPO": "Exited / Public",
    "Acquired": "Exited / Public",
    "Subsidiary": "Exited / Public",

    "Private": "Private / Other",
    "Private Equity": "Private / Other",

    "Unknown": "Unclassified"
}

df["stage_group"] = df["stage"].map(stage_group_mapping)
df["stage_group"] = df["stage_group"].fillna("Private / Other")

In [23]:
# Count layoff events by year and stage group

stage_year_events = (
    df.groupby(["year", "stage_group"])
    .size()
    .reset_index(name="layoff_events")
)

stage_year_events

,year,stage_group,layoff_events
0,2020,Early Stage,122
1,2020,Exited / Public,58
2,2020,Late Stage & Growth,139
3,2020,Private / Other,5
4,2020,Unclassified,54
5,2021,Early Stage,4
6,2021,Exited / Public,8
7,2021,Late Stage & Growth,6
8,2021,Unclassified,6
9,2022,Early Stage,174


In [24]:
# Create clustered bar chart

fig = px.bar(
    stage_year_events,
    x="year",
    y="layoff_events",
    color="stage_group",
    barmode="group",
    title="Frequency of Layoff Events by Funding Stage Group and Year",
    labels={
        "year": "Year",
        "layoff_events": "Number of Layoff Events",
        "stage_group": "Funding Stage Group"
    },
    text="layoff_events"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    height=600,
    plot_bgcolor="#F8F9FA",
    paper_bgcolor="#EAF2F8",
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray"
    ),
    legend_title_text="Funding Stage Group"
)

fig.show()

In [25]:
# Sort data
stage_year_events_sorted = stage_year_events.sort_values(["year", "stage_group"])

fig = px.bar(
    stage_year_events_sorted,
    x="stage_group",
    y="layoff_events",
    color="stage_group",
    animation_frame="year",
    text="layoff_events",
    title="Animated Layoff Event Frequency by Funding Stage Group",
    labels={
        "stage_group": "Funding Stage Group",
        "layoff_events": "Number of Layoff Events",
        "year": "Year"
    },
    color_discrete_sequence=px.colors.qualitative.Bold,
    range_y=[0, stage_year_events["layoff_events"].max() * 1.25],
    hover_data={
        "stage_group": True,
        "layoff_events": ":,.0f",
        "year": True
    }
)

fig.update_traces(
    textposition="outside",
    textfont=dict(size=13, color="black"),
    marker_line_width=1.5,
    marker_line_color="black",
    opacity=0.9,
    hovertemplate=
        "<b>%{x}</b><br>" +
        "Layoff Events: %{y:,.0f}<br>" +
        "<extra></extra>"
)

fig.update_layout(
    title=dict(
        text=(
            "Animated Layoff Events by Funding Stage Group"
            "<br><sup>Year-by-year comparison of layoffs across funding stages</sup>"
        ),
        x=0.5,
        font=dict(size=25, family="Arial Black")
    ),

    height=700,
    width=1000,

    plot_bgcolor="#F8FAFC",
    paper_bgcolor="#EAF2F8",

    xaxis=dict(
        title="Funding Stage Group",
        showgrid=False,
        tickangle=-20,
        showline=True,
        linewidth=2,
        linecolor="black"
    ),

    yaxis=dict(
        title="Number of Layoff Events",
        showgrid=True,
        gridcolor="#D6DBDF",
        zeroline=True,
        zerolinecolor="black",
        showline=True,
        linewidth=2,
        linecolor="black"
    ),

    showlegend=False,

    font=dict(
        size=13,
        family="Arial"
    ),

    bargap=0.25,

    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Arial"
    ),

    transition=dict(
        duration=700,
        easing="cubic-in-out"
    ),

    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.05,
            y=1.15,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        {
                            "frame": {"duration": 900, "redraw": True},
                            "transition": {"duration": 600},
                            "fromcurrent": True
                        }
                    ]
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }
                    ]
                )
            ]
        )
    ]
)

fig.show()

## **📋 Summary of Key Findings**

| Insight | Finding |
|---------|---------|
| Most affected industry | *Retail* |
| Worst year | *2023 Specifically Q1* |
| AI vs Non-AI resilience | *AI Companies are more resilient* |
| Highest layoff country | *United States, Canada is the 8th* |
| Most vulnerable stage | *Post-Ipo* |